# 01 · Sudan data: price, exchange rate, the gap and the split

Reads `data/processed/v2/sudan_merged.csv` (built from the author's `data/raw/new_Data.csv` by the pipeline),
`results/summary/facts.json` and `results/data/split_summary.json`. The global track's raw data are
Yahoo Finance / FRED downloads that are not redistributed, so no global data figure is drawn here.

**In this public repository the Sudan data are not included** (see `data/raw/SUDAN_DATA.md`; available on request),
so this notebook is stored without outputs and runs only with the full file.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
load = lambda p: json.loads((ROOT / p).read_text())
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3, "axes.spines.top": False,
                     "axes.spines.right": False})

In [ ]:
df = pd.read_csv(ROOT / "data/processed/v2/sudan_merged.csv", parse_dates=["date"]).set_index("date")
facts = load("results/summary/facts.json")
splits = load("results/data/split_summary.json")
print(len(df), "daily rows;", int((df.is_ffilled == 1).sum()), "forward-filled;", int(df.in_long_gap.sum()), "in the 404-day gap")

## Price and exchange rate

Top: local price and the 70/15/15 split. Bottom: SDG/USD. The local price moves almost one-for-one with the
official exchange rate (R² of log price on log SDG/USD, printed below).

In [ ]:
n = len(df); b1, b2 = df.index[int(0.70 * n)], df.index[int(0.70 * n) + int(0.15 * n)]
fig, ax = plt.subplots(2, 1, figsize=(10, 6.5), sharex=True, gridspec_kw={"height_ratios": [3, 2]})
obs = df[df.is_ffilled == 0]
ax[0].plot(df.index, df.price_sdg, lw=0.8, color="0.6", label="daily (forward-filled)")
ax[0].plot(obs.index, obs.price_sdg, ".", ms=1.5, color="C0", label="observed")
ax[0].axhline(facts["range"]["sudan"]["train_max"], ls=":", color="C3", lw=1, label="training maximum")
ax[0].set_yscale("log"); ax[0].set_ylabel("SDG / g (log)")
ax[1].plot(df.index, df.fx_rate, color="C1", lw=1); ax[1].set_yscale("log"); ax[1].set_ylabel("SDG / USD (log)")
for a in ax:
    a.axvspan(df.index[df.in_long_gap == 1].min(), df.index[df.in_long_gap == 1].max(), color="0.9", lw=0)
    for d, c in ((b1, "k"), (b2, "k")):
        a.axvline(d, color=c, lw=0.8, ls="--")
    for d in ("2019-04-11", "2021-10-25"):
        a.axvline(pd.Timestamp(d), color="C3", lw=0.8, alpha=0.6)
ax[0].legend(loc="upper left", fontsize=8)
ax[0].set_title("Shaded: 404-day gap. Dashed: train / val / test boundaries. Red: 2019-04-11 and 2021-10-25")
plt.tight_layout(); plt.show()

In [ ]:
r, v = facts["range"]["sudan"], facts["variance"]
print(f"test targets / training maximum: {r['test_min_over_train_max']:.1f} .. {r['test_max_over_train_max']:.1f}")
print(f"R² log price on log SDG/USD (observed rows): {v['r2_logprice_on_logfx_observed']:.4f}")

## Windows per split

Windows touching the gap are dropped, which leaves the Sudan validation set small and confined to 2021-03 → 2021-10.

In [ ]:
w = pd.DataFrame(splits["splits"])
w["dates"] = w["row_range"].map(lambda r: f"{r[0]} → {r[1]}")
w.pivot_table(index=["track", "horizon"], columns="split", values="windows", aggfunc="first")[["train", "val", "test"]]